# Data preparation

Run from `task2/`. Reuse saved files; all checks run in memory.

## Setup and imports


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

COLUMNS = ["compound_iso_smiles", "target_sequence", "affinity"]
PAIR_KEYS = ["compound_iso_smiles", "target_sequence"]
DAVIS_SOURCE = Path("../data/davis_all.csv")
KIBA_SOURCE = Path("../data/kiba_clean.csv")
DAVIS_OUTPUT = Path("data/davis_clean_task2.csv")


## DAVIS preprocessing

Remove invalid rows and average duplicate drug–protein pairs. DAVIS already contains pKd values, so no log conversion is needed.


In [2]:
davis_raw = pd.read_csv(DAVIS_SOURCE, usecols=COLUMNS)[COLUMNS]
davis = davis_raw.copy()
davis["affinity"] = pd.to_numeric(davis["affinity"], errors="coerce")
valid_scores = np.isfinite(davis["affinity"])
print(f"DAVIS rows with invalid/non-finite scores removed: {(~valid_scores).sum():,}")
davis = davis.loc[valid_scores].copy()

for column in PAIR_KEYS:
    davis[column] = davis[column].astype("string").str.strip()
valid_sequences = davis[PAIR_KEYS].notna().all(axis=1) & davis[PAIR_KEYS].ne("").all(axis=1)
print(f"DAVIS rows with missing/blank identifiers removed: {(~valid_sequences).sum():,}")
davis = davis.loc[valid_sequences].copy()

rows_before_deduplication = len(davis)
davis = davis.groupby(PAIR_KEYS, as_index=False, sort=False)["affinity"].mean()
print(f"DAVIS repeated pair rows combined: {rows_before_deduplication - len(davis):,}")

saved_davis = pd.read_csv(DAVIS_OUTPUT, float_precision="round_trip")
pd.testing.assert_frame_equal(davis, saved_davis, check_dtype=False, check_exact=True)
print("Existing cleaned DAVIS matches preprocessing; no CSV written.")


DAVIS rows with invalid/non-finite scores removed: 0
DAVIS rows with missing/blank identifiers removed: 0
DAVIS repeated pair rows combined: 4,284
Existing cleaned DAVIS matches preprocessing; no CSV written.


## Load KIBA

Use the existing cleaned data from task 1 without changing its scores.


In [3]:
kiba = pd.read_csv(KIBA_SOURCE)
print(f"Loaded KIBA directly from {KIBA_SOURCE}")


Loaded KIBA directly from ../data/kiba_clean.csv


## Dataset summary

Check counts, score ranges and duplicate pairs.


In [4]:
def verify_dataset(frame, name):
    assert set(COLUMNS).issubset(frame.columns), f"{name}: missing required columns"
    assert not frame.empty, f"{name}: no observations"
    assert pd.api.types.is_numeric_dtype(frame["affinity"]), f"{name}: scores must be numeric"
    assert np.isfinite(frame["affinity"].to_numpy()).all(), f"{name}: invalid scores"
    for column in PAIR_KEYS:
        identifiers = frame[column].astype("string").str.strip()
        assert identifiers.notna().all() and identifiers.ne("").all(), f"{name}: invalid {column}"
    duplicates = int(frame.duplicated(PAIR_KEYS).sum())
    summary = {
        "Dataset": name,
        "Observations": len(frame),
        "Unique drugs": frame[PAIR_KEYS[0]].nunique(),
        "Unique proteins": frame[PAIR_KEYS[1]].nunique(),
        "Minimum affinity": frame["affinity"].min(),
        "Maximum affinity": frame["affinity"].max(),
        "Duplicate pairs remaining": duplicates,
    }
    for label, value in summary.items():
        print(f"{label}: {value}")
    print()
    assert duplicates == 0, f"{name}: duplicate pairs remain"
    return summary

summaries = [verify_dataset(davis, "DAVIS"), verify_dataset(kiba, "KIBA")]
assert davis["affinity"].between(4.99, 10.81).all(), "DAVIS: unexpected pKd range"
print("Verification passed: DAVIS has a sensible pKd range; both datasets have valid scores.")
pd.DataFrame(summaries).set_index("Dataset")


Dataset: DAVIS
Observations: 25772
Unique drugs: 68
Unique proteins: 379
Minimum affinity: 5.0
Maximum affinity: 10.619788758288394
Duplicate pairs remaining: 0

Dataset: KIBA
Observations: 117657
Unique drugs: 2068
Unique proteins: 229
Minimum affinity: 0.0
Maximum affinity: 17.200179498
Duplicate pairs remaining: 0

Verification passed: DAVIS has a sensible pKd range; both datasets have valid scores.


,Observations,Unique drugs,Unique proteins,Minimum affinity,Maximum affinity,Duplicate pairs remaining
Dataset,,,,,,
DAVIS,25772,68,379,5.0,10.619789,0
KIBA,117657,2068,229,0.0,17.200179,0


## Train/validation/test split

Split observed pairs into 70% training, 15% validation and 15% testing using seed 42. Training fits the model, validation helps tune it, and testing evaluates it.

Reserve pairs covering every drug and protein in training so CF can evaluate known entities. Split the remaining pairs to reach the target ratios. Missing KIBA observations are not filled with zero.


In [5]:
SEED = 42
SPLIT_DIR = Path("data/splits")

# Load cleaned data
datasets = {
    "davis": pd.read_csv(DAVIS_OUTPUT, usecols=COLUMNS)[COLUMNS],
    "kiba": pd.read_csv(KIBA_SOURCE, usecols=COLUMNS)[COLUMNS],
}


def coverage_aware_split(frame, seed=42):
    """Keep every drug and protein in training."""
    assert not frame.duplicated(PAIR_KEYS).any()
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(frame))
    training_mask = np.zeros(len(frame), dtype=bool)
    seen_drugs, seen_proteins = set(), set()
    pairs = list(frame[PAIR_KEYS].itertuples(index=False, name=None))
    for index in order:
        drug, protein = pairs[index]
        if drug not in seen_drugs or protein not in seen_proteins:
            training_mask[index] = True
            seen_drugs.add(drug)
            seen_proteins.add(protein)

    training_size = max(int(0.70 * len(frame)), int(training_mask.sum()))
    remaining = order[~training_mask[order]]
    extra_training = training_size - int(training_mask.sum())
    training_mask[remaining[:extra_training]] = True
    evaluation = remaining[extra_training:]
    validation_size = min(int(0.15 * len(frame)), len(evaluation) // 2)
    indices = {
        "train": order[training_mask[order]],
        "val": evaluation[:validation_size],
        "test": evaluation[validation_size:],
    }
    return {name: frame.iloc[index].reset_index(drop=True) for name, index in indices.items()}


def verify_splits(source, partitions):
    """Check overlap, coverage and scores."""
    pair_sets = {
        name: set(part[PAIR_KEYS].itertuples(index=False, name=None))
        for name, part in partitions.items()
    }
    for name, part in partitions.items():
        assert not part.empty, f"Empty {name} partition"
        assert len(pair_sets[name]) == len(part), f"Duplicates in {name}"
        assert np.isfinite(part["affinity"]).all()
    assert pair_sets["train"].isdisjoint(pair_sets["val"])
    assert pair_sets["train"].isdisjoint(pair_sets["test"])
    assert pair_sets["val"].isdisjoint(pair_sets["test"])
    source_pairs = set(source[PAIR_KEYS].itertuples(index=False, name=None))
    assert set.union(*pair_sets.values()) == source_pairs
    assert sum(map(len, partitions.values())) == len(source)
    for name in ["val", "test"]:
        for column in PAIR_KEYS:
            assert set(partitions[name][column]) <= set(partitions["train"][column])
    combined = pd.concat(partitions.values(), ignore_index=True)
    pd.testing.assert_frame_equal(
        source.set_index(PAIR_KEYS).sort_index(),
        combined.set_index(PAIR_KEYS).sort_index(),
        check_exact=True,
    )


splits = {}
split_rows = []
for dataset_name, frame in datasets.items():
    partitions = coverage_aware_split(frame, SEED)
    verify_splits(frame, partitions)
    repeated = coverage_aware_split(frame, SEED)
    loaded = {}
    for partition_name, part in partitions.items():
        pd.testing.assert_frame_equal(part, repeated[partition_name], check_exact=True)
        output_path = SPLIT_DIR / f"{dataset_name}_{partition_name}.csv"
        # Preserve float precision when reloading
        loaded[partition_name] = pd.read_csv(output_path, float_precision="round_trip")
        pd.testing.assert_frame_equal(part, loaded[partition_name], check_exact=True)
        split_rows.append({
            "Dataset": dataset_name.upper(), "Split": partition_name,
            "Observations": len(part), "Percentage": 100 * len(part) / len(frame),
        })
    verify_splits(frame, loaded)
    splits[dataset_name] = partitions

split_summary = pd.DataFrame(split_rows)
print(split_summary.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Split checks passed, including reproducibility and exact score preservation.")


Dataset Split  Observations  Percentage
  DAVIS train         18040     69.9984
  DAVIS   val          3865     14.9969
  DAVIS  test          3867     15.0047
   KIBA train         82359     69.9992
   KIBA   val         17648     14.9995
   KIBA  test         17650     15.0012
Split checks passed, including reproducibility and exact score preservation.
